Read the [accompanying blog post](https://acytoo.com/ladder/llm-tut-3-nano-gpt-multimodal/) for explanations and experiment results.

# GPT2 implementation in Pytorch
require Pytorch > 2.10

https://github.com/karpathy/nanoGPT

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

### Model blocks

In [ ]:
class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


### GPT2

In [ ]:
@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float
    bias: bool

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # with weight tying when using torch.compile() some warnings get generated:
        # "UserWarning: functional_call was passed multiple values for tied weights.
        # This behavior is deprecated and will be an error in future versions"
        # not 100% sure what this is, so far seems to be harmless. TODO investigate
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss

    def crop_block_size(self, block_size):
        # model surgery to decrease the block size if necessary
        # e.g. we may load the GPT2 pretrained model checkpoint (block size 1024)
        # but want to use a smaller block size for some smaller, simpler model
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # start with all of the candidate parameters
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # filter out those that do not require grad
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        use_fused = (device_type == 'cuda')
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")

        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        """ estimate model flops utilization (MFU) in units of A100 bfloat16 peak FLOPS """
        # first estimate the number of flops we do per iteration.
        # see PaLM paper Appendix B as ref: https://arxiv.org/abs/2204.02311
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd//cfg.n_head, cfg.block_size
        flops_per_token = 6*N + 12*L*H*Q*T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        # express our flops throughput as ratio of A100 bfloat16 peak flops
        flops_achieved = flops_per_iter * (1.0/dt) # per second
        flops_promised = 312e12 # A100 GPU bfloat16 peak flops is 312 TFLOPS
        mfu = flops_achieved / flops_promised
        return mfu

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# Train our new model

### Prepare MIDI input and Chinese input, and their en/decoders

In [ ]:
!pip install -q pretty_midi mido

import io
import urllib.request
import zipfile
from pathlib import Path
import mido
import pretty_midi
import json

# Download & extract Satie / Joplin MIDI archive from llm_tut3.1
ENGLISH_BOOK = "little_brother"

archive_path = Path("toy_multimodal_dataset_v1.1.zip")
data_dir = Path("toy_multimodal_dataset_v1.1")

if not archive_path.exists():
    urllib.request.urlretrieve(
        "https://dl.acytoo.com/toy_multimodal_dataset_v1.1.zip",
        archive_path,
    )

with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(".")

# MIDI encoder/decoder from llm_tut3.1
PITCH_MIN, PITCH_MAX = 21, 108

def encode_midi(midi_path: str) -> list[int]:
    midi = mido.MidiFile(midi_path)
    midi.tracks = [mido.merge_tracks(midi.tracks)]
    midi.type = 0
    buf = io.BytesIO()
    midi.save(file=buf)
    buf.seek(0)
    pm = pretty_midi.PrettyMIDI(buf)

    events = []
    for inst in pm.instruments:
        if not inst.is_drum:
            for note in inst.notes:
                pitch = max(PITCH_MIN, min(PITCH_MAX, note.pitch)) - PITCH_MIN
                events.append((note.start, 1, pitch, note.velocity))
                events.append((note.end, 0, pitch, 0))
    events.sort(key=lambda x: (x[0], x[1]))

    tokens, current_time, current_vel_bin = [], 0.0, -1
    for time, etype, pitch, vel in events:
        dt_steps = int(round((time - current_time) * 100))
        while dt_steps > 0:
            step = min(dt_steps, 100)
            tokens.append(176 + (step - 1))
            dt_steps -= step
        current_time = time
        if etype == 1:
            vel_bin = vel // 4
            if vel_bin != current_vel_bin:
                tokens.append(276 + vel_bin)
                current_vel_bin = vel_bin
            tokens.append(pitch)
        else:
            tokens.append(88 + pitch)
    return tokens

def decode_midi(tokens: list[int], out_path: str = "generated.mid"):
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    current_time, current_vel, active_notes = 0.0, 80, {}
    for tok in tokens:
        if 0 <= tok < 88:
            active_notes[tok + PITCH_MIN] = (current_time, current_vel)
        elif 88 <= tok < 176:
            pitch = (tok - 88) + PITCH_MIN
            if pitch in active_notes:
                start, vel = active_notes.pop(pitch)
                if current_time > start:
                    piano.notes.append(
                        pretty_midi.Note(velocity=vel, pitch=pitch, start=start, end=current_time)
                    )
        elif 176 <= tok < 276:
            current_time += ((tok - 176) + 1) * 0.01
        elif 276 <= tok < 308:
            current_vel = (tok - 276) * 4 + 2
    pm.instruments.append(piano)
    pm.write(out_path)
    return pm

# Encode Satie and Joplin MIDI streams
# Read the predefined train/validation splits
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


splits = ("train", "val")

text_corpus = {
    split: {
        "zh": (
            data_dir / "text/zh" / f"{split}_zh.txt"
        ).read_text(encoding="utf-8"),
        "en": (
            data_dir / "text/en" / ENGLISH_BOOK / f"{split}_en.txt"
        ).read_text(encoding="utf-8"),
    }
    for split in splits
}

music_manifest = read_jsonl(data_dir / "music/manifest.jsonl")

pair_rows = {
    split: read_jsonl(data_dir / "pairs" / f"{split}.jsonl")
    for split in splits
}

work_captions = read_jsonl(data_dir / "captions/works.jsonl")


# Keep the existing MIDI and special token IDs
TOK_TEXT, TOK_MUSIC, TOK_EOS, TOK_UNK = 308, 309, 310, 311
CHAR_OFFSET = 312

DESCRIBE_PROMPTS = {
    "zh": "用中文描述这段音乐：",
    "en": "Describe this music in English:",
}

vocab_text = (
    text_corpus["train"]["zh"]
    + text_corpus["train"]["en"]
    + "".join(
        row["caption_zh"] + row["caption_en"]
        for row in pair_rows["train"]
    )
    + "".join(DESCRIBE_PROMPTS.values())
)

all_chars = sorted(set(vocab_text))
stoi = {ch: CHAR_OFFSET + i for i, ch in enumerate(all_chars)}
itos = {token: ch for ch, token in stoi.items()}

raw_vocab_size = CHAR_OFFSET + len(all_chars)
vocab_size = ((raw_vocab_size + 63) // 64) * 64


def encode_zh(text: str) -> list[int]:
    return [stoi.get(ch, TOK_UNK) for ch in text]


def decode_zh(tokens: list[int]) -> str:
    return "".join(itos[t] for t in tokens if t in itos)


encode_text = encode_zh
decode_text = decode_zh


text_data = {
    split: {
        lang: encode_text(text)
        for lang, text in text_corpus[split].items()
    }
    for split in splits
}

music_data = {
    split: [
        encode_midi(str(data_dir / row["midi_file"]))
        for row in music_manifest
        if row["split"] == split
    ]
    for split in splits
}

paired_data = {
    split: [
        {
            "clip_id": row["clip_id"],
            "music": encode_midi(str(data_dir / row["midi_file"])),
            "zh": encode_text(row["caption_zh"]),
            "en": encode_text(row["caption_en"]),
        }
        for row in pair_rows[split]
    ]
    for split in splits
}


print(f"English book: {ENGLISH_BOOK}")
print(
    f"Unique text chars: {len(all_chars):,} | "
    f"Unified vocab_size: {vocab_size:,}"
)

for split in splits:
    print(
        f"{split}: Chinese {len(text_data[split]['zh']):,} tokens | "
        f"English {len(text_data[split]['en']):,} tokens | "
        f"Chopin {len(music_data[split])} works, "
        f"{sum(map(len, music_data[split])):,} MIDI tokens | "
        f"{len(paired_data[split])} paired clips"
    )

max_pair_tokens = max(
    len(encode_text(DESCRIBE_PROMPTS[lang]))
    + len(pair["music"])
    + len(pair[lang])
    + 4
    for split in splits
    for pair in paired_data[split]
    for lang in ("zh", "en")
)

print(f"Longest full music-description sample: {max_pair_tokens} tokens")

English book: little_brother
Unique text chars: 4,576 | Unified vocab_size: 4,928
train: Chinese 650,317 tokens | English 504,406 tokens | Chopin 12 works, 63,776 MIDI tokens | 488 paired clips
val: Chinese 72,324 tokens | English 65,325 tokens | Chopin 2 works, 8,736 MIDI tokens | 71 paired clips
Longest full music-description sample: 326 tokens


In [ ]:
import random
from contextlib import nullcontext
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
block_size = 512
batch_size = 32
max_iters = 1000
eval_interval = 200
eval_iters = 3

torch.manual_seed(1337)
train_rng = random.Random(1337)
TASKS = ("zh", "en", "music", "text_to_music", "music_to_text")
TASK_WEIGHTS = (0.30, 0.35, 0.15, 0.10, 0.10)

# Preserve the train/validation splits loaded in cell 9.
music_streams = {
    split: [tokens + [TOK_EOS] for tokens in music_data[split]]
    for split in ("train", "val")
}
describe_prefix = {
    lang: encode_text(prompt) for lang, prompt in DESCRIBE_PROMPTS.items()
}
assert max_pair_tokens - 1 <= block_size, "Increase block_size to fit complete pairs."


def get_multimodal_batch(split, task=None, rng=None):
    rng = train_rng if rng is None else rng
    if task is None:
        task = rng.choices(TASKS, weights=TASK_WEIGHTS, k=1)[0]

    # Use one task per batch so short caption targets get dedicated updates.
    x = torch.full((batch_size, block_size), TOK_EOS, dtype=torch.long)
    y = torch.full((batch_size, block_size), -1, dtype=torch.long)

    for i in range(batch_size):
        if task in ("zh", "en", "music"):
            if task == "music":
                tokens = rng.choice(music_streams[split])
                prefix = [TOK_MUSIC]
            else:
                tokens = text_data[split][task]
                prefix = [TOK_TEXT]
            start = rng.randint(0, max(0, len(tokens) - block_size))
            target = tokens[start:start + block_size]
        else:
            pair = rng.choice(paired_data[split])
            lang = rng.choice(("zh", "en"))
            if task == "text_to_music":
                prefix = [TOK_TEXT] + pair[lang] + [TOK_MUSIC]
                target = pair["music"] + [TOK_EOS]
            elif task == "music_to_text":
                prefix = (
                    [TOK_TEXT] + describe_prefix[lang] + [TOK_MUSIC]
                    + pair["music"] + [TOK_TEXT]
                )
                target = pair[lang] + [TOK_EOS]
            else:
                raise ValueError(f"Unknown task: {task}")

        sequence = torch.tensor(prefix + target, dtype=torch.long)
        n = len(sequence) - 1
        if n > block_size:
            raise ValueError("Sample exceeds block_size; do not truncate paired data.")
        x[i, :n] = sequence[:-1]
        y[i, :n] = sequence[1:]
        # GPT.forward already uses ignore_index=-1: ignore prompts and padding.
        y[i, :len(prefix) - 1] = -1

    return x.to(device), y.to(device)


# Keep the existing GPT architecture.
config = GPTConfig(
    block_size=block_size, vocab_size=vocab_size,
    n_layer=6, n_head=6, n_embd=384, dropout=0.1, bias=False,
)
model = GPT(config).to(device)
optimizer = model.configure_optimizers(
    weight_decay=0.1, learning_rate=1e-3,
    betas=(0.9, 0.95), device_type=device,
)

amp_enabled = device == "cuda"
# dtype = (
#     torch.bfloat16
#     if amp_enabled and torch.cuda.is_bf16_supported()
#     else torch.float16
# )

dtype = torch.float16  # Use native FP16 on T4.
scaler = torch.amp.GradScaler(
    "cuda", enabled=amp_enabled and dtype == torch.float16,
)


def amp_context():
    if amp_enabled:
        return torch.amp.autocast(device_type="cuda", dtype=dtype)
    return nullcontext()


@torch.no_grad()
def estimate_loss():
    was_training = model.training
    model.eval()
    results = {}
    # Repeat the same validation samples without changing the training RNG.
    val_rng = random.Random(2026)
    for task in TASKS:
        losses = []
        for _ in range(eval_iters):
            xb, yb = get_multimodal_batch("val", task, val_rng)
            with amp_context():
                _, loss = model(xb, yb)
            losses.append(loss.item())
        results[task] = sum(losses) / len(losses)
    model.train(was_training)
    return results


number of parameters: 12.51M
num decayed parameter tensors: 26, with 12,705,792 parameters
num non-decayed parameter tensors: 13, with 4,992 parameters
using fused AdamW: True


### Train

In [ ]:
TASK_WEIGHTS = (0.30, 0.35, 0.15, 0.10, 0.10)
max_iters = 2000

for group in optimizer.param_groups:
    group["lr"] = 1e-4

model.train()

for step in range(1, max_iters + 1):
    task = train_rng.choices(TASKS, weights=TASK_WEIGHTS, k=1)[0]
    xb, yb = get_multimodal_batch("train", task)
    optimizer.zero_grad(set_to_none=True)
    with amp_context():
        _, loss = model(xb, yb)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if step == 1 or step % eval_interval == 0 or step == max_iters:
        val_losses = estimate_loss()
        val_log = " | ".join(f"{k}={v:.3f}" for k, v in val_losses.items())
        print(f"step {step:4d} | train({task})={loss.item():.3f} | val: {val_log}")

step    1 | train(text_to_music)=0.896 | val: zh=4.174 | en=1.609 | music=5.288 | text_to_music=4.951 | music_to_text=0.093
step  200 | train(en)=1.576 | val: zh=4.137 | en=1.514 | music=5.425 | text_to_music=5.012 | music_to_text=0.081
step  400 | train(en)=1.538 | val: zh=4.129 | en=1.495 | music=5.520 | text_to_music=5.103 | music_to_text=0.084
step  600 | train(music_to_text)=0.047 | val: zh=4.128 | en=1.489 | music=5.621 | text_to_music=5.185 | music_to_text=0.103
step  800 | train(zh)=3.321 | val: zh=4.121 | en=1.466 | music=5.731 | text_to_music=5.251 | music_to_text=0.088
step 1000 | train(music)=0.463 | val: zh=4.124 | en=1.452 | music=5.729 | text_to_music=5.280 | music_to_text=0.100
step 1200 | train(music)=0.433 | val: zh=4.117 | en=1.445 | music=5.735 | text_to_music=5.305 | music_to_text=0.090
step 1400 | train(music_to_text)=0.041 | val: zh=4.128 | en=1.437 | music=5.818 | text_to_music=5.374 | music_to_text=0.089
step 1600 | train(text_to_music)=0.624 | val: zh=4.119 | 

In [ ]:
!apt-get install -y -qq fluidsynth
!pip install -q pyfluidsynth

import importlib
import pretty_midi.fluidsynth

# Refresh FluidSynth detection without restarting the runtime.
importlib.reload(pretty_midi.fluidsynth)
import torch
from IPython.display import Audio, display

Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../00-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../03-libxtst6_2%3a1.2.3-1.1ubuntu0.1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1.1ubuntu0.1) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../04-session-migration_0.3.9build1_amd64.deb ...
Unpacking session-migration (0.3.

In [ ]:
SEED = 42
TEMPERATURE = 0.8
TOP_K = 40
SAMPLE_INDEX = 0
LANG = "zh"  # "zh" or "en" for paired tasks

sample_device = next(model.parameters()).device
sample_rng = torch.Generator(device=sample_device).manual_seed(SEED)


@torch.inference_mode()
def sample_tokens(prompt_ids, output_type, max_new_tokens):
    was_training = model.training
    model.eval()
    ids = torch.tensor([prompt_ids], dtype=torch.long, device=sample_device)
    allowed = list(range(308)) if output_type == "music" else list(itos)
    allowed.append(TOK_EOS)
    token_mask = torch.full(
        (model.config.vocab_size,), -float("inf"), device=sample_device,
    )
    token_mask[allowed] = 0
    generated = []

    try:
        for _ in range(max_new_tokens):
            with amp_context():
                logits, _ = model(ids[:, -model.config.block_size:])
            # Restrict sampling to the requested output modality.
            logits = logits[:, -1, :].float() / TEMPERATURE + token_mask
            cutoff = torch.topk(logits, min(TOP_K, len(allowed))).values[:, -1:]
            logits = logits.masked_fill(logits < cutoff, -float("inf"))
            next_id = torch.multinomial(
                torch.softmax(logits, dim=-1), 1, generator=sample_rng,
            )
            token = next_id.item()
            if token == TOK_EOS:
                break
            generated.append(token)
            ids = torch.cat((ids, next_id), dim=1)
    finally:
        model.train(was_training)

    return generated


def preview_midi(tokens, filename):
    midi = decode_midi(tokens, filename)
    note_count = sum(len(inst.notes) for inst in midi.instruments)
    print(f"Saved: {filename} | Complete notes: {note_count}")
    if note_count:
        # Built-in sine-wave preview; the MIDI file uses a piano instrument.
        display(Audio(midi.fluidsynth(fs=44100), rate=44100))
    else:
        print("No complete notes to play.")


# Chinese and English continuation
for label, prompt in (("Chinese", "悟空道："), ("English", "Jack looked")):
    print(f"\n=== {label} continuation ===")
    output = sample_tokens([TOK_TEXT] + encode_text(prompt), "text", 200)
    print(prompt + decode_text(output))

# Unconditional music generation
print("\n=== Music generation ===")
music_output = sample_tokens([TOK_MUSIC], "music", 512)
preview_midi(music_output, "generated_chopin.mid")

# Use a held-out clip for both paired tasks.
pair = paired_data["val"][SAMPLE_INDEX]
reference_caption = decode_text(pair[LANG])
print(f"\n=== Validation clip: {pair['clip_id']} ===")
print("Reference description:", reference_caption)
preview_midi(pair["music"], "reference_clip.mid")

# Text -> music
print("\n=== Text -> music ===")
prompt_ids = [TOK_TEXT] + pair[LANG] + [TOK_MUSIC]
conditioned_music = sample_tokens(prompt_ids, "music", 256)
preview_midi(conditioned_music, "text_to_music.mid")

# Music -> text: the reference caption is not included in this prompt.
print("\n=== Music -> text ===")
prompt_ids = (
    [TOK_TEXT] + encode_text(DESCRIBE_PROMPTS[LANG])
    + [TOK_MUSIC] + pair["music"] + [TOK_TEXT]
)
description = sample_tokens(prompt_ids, "text", 160)
print("Model description:", decode_text(description))
print("Reference description:", reference_caption)


=== Chinese continuation ===
悟空道：「那和尚，你好，你叫我出來也。」公主即引八戒牽馬挑擔，找大路而入。那師父，一程遙觀看，只見一座高山門，忽見一座高山，有六里遠近，卻是一座高山，嶺上有一石碑，石崖石碣，右有千千萬丈之遙。石叢叢，有兩三千條觀，乃是「幽平。」三藏道：「有六里。」高僧道：「那廂風的是一座石橋，橋邊有青松樹乃是石石猴石邊，石橋，石叢雜樹梢頭石，石石橋，石叢叢叢，石叢叢叢叢。橋邊有石石磷隱有松，石，」；石叢，石叢叢叢叢叢，石

=== English continuation ===
Jack looked to up it and the people we were notaking the alme walks and the cared of and coursent and droned scripors. The timed to talking about the fater truff or and the dodes to had and the stranorm were sur

=== Music generation ===
Saved: generated_chopin.mid | Complete notes: 134



=== Validation clip: chopin_op37_no01_f702fbb2_0000 ===
Reference description: 这段钢琴的音高范围是C2至D#5，音符起音密度中等，MIDI力度整体中等。
Saved: reference_clip.mid | Complete notes: 48



=== Text -> music ===
Saved: text_to_music.mid | Complete notes: 16



=== Music -> text ===
Model description: 这段钢琴的音高范围是E2至G5，音符起音较密集，MIDI力度整体中等。
Reference description: 这段钢琴的音高范围是C2至D#5，音符起音密度中等，MIDI力度整体中等。
